# Customer Support on Twitter ? Exploratory Dataset Analysis
### Hiver SDE Intern Take-Home Project: Empirical Brand Selection & Intent Discovery

This notebook walks through the data analysis, brand ranking, thread reconstruction, and intent discovery conducted on the Kaggle Customer Support on Twitter dataset (`thoughtvector/customer-support-on-twitter`).

In [ ]:
import os
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Load dataset overview metrics
with open('../data_analysis/dataset_overview.json') as f:
    overview = json.load(f)

print(f"Total Raw Records: {overview['total_rows']:,}")
print(f"Customer Queries (inbound=True): {overview['inbound_distribution']['True']:,}")
print(f"Support Brand Replies (inbound=False): {overview['inbound_distribution']['False']:,}")
print(f"Total Unique Brands: {overview['total_brands']}")

## 1. Top Brands by Support Tweet Volume

In [ ]:
top_brands = pd.Series(overview['top_20_brands'])
plt.figure(figsize=(10, 5))
top_brands.head(10).plot(kind='barh', color='royalblue')
plt.title('Top 10 Brands by Support Volume on Twitter')
plt.xlabel('Number of Support Tweets')
plt.gca().invert_yaxis()
plt.grid(axis='x', linestyle='--', alpha=0.7)
plt.show()

## 2. Empirical Brand Ranking & DM Deflection Analysis

Why AmazonHelp was selected over AppleSupport, Uber, and Delta:
* **AmazonHelp DM Deflection**: Only 0.82% (over 99% of replies provide substantive public guidance).
* **AppleSupport DM Deflection**: 52.56% (majority say *'Please DM us'*).
* **TMobileHelp DM Deflection**: 82.12%.

In [ ]:
with open('../data_analysis/brand_analysis.json') as f:
    brands = json.load(f)

df_brands = pd.DataFrame(brands)
df_brands[['brand', 'total_support_tweets', 'reconstructed_pairs', 'dm_deflection_rate_pct', 'suitability_score']]

## 3. Discovered Intent Topics via NMF Topic Modeling

In [ ]:
with open('../data_analysis/discovered_topics.json') as f:
    topics = json.load(f)

for t in topics:
    print(f"Topic {t['topic_id'] + 1}: {', '.join(t['top_words'][:8])}")

## 4. Benchmark Results on 200-Sample Frozen Golden Set

In [ ]:
with open('../experiments/evaluation_results.json') as f:
    results = json.load(f)

print("=== Intent Classification Benchmark ===")
for model_name, metrics in results['intent_classification'].items():
    print(f"{model_name:<30} | Accuracy: {metrics['accuracy']:.4f} | Macro-F1: {metrics['macro_f1']:.4f}")

print("\n=== Retrieval & Safety Escalation ===")
print(f"Retrieval Recall@5: {results['retrieval']['recall_at_5']:.4f}")
print(f"Retrieval MRR:      {results['retrieval']['mrr']:.4f}")
print(f"Escalation F1:      {results['escalation']['escalate_f1']:.4f}")
print(f"False Auto-Handle:  {results['escalation']['false_auto_handle_rate']*100:.1f}%")